# 04 규정 기반 RAG — 검색 노트북

**하는 일**: 검사 항목과 광고 상황을 문장으로 넣으면, 근거가 되는 규정 카드 5장을 찾아줍니다. 05 LLM 판단은 여기서 찾은 카드만 근거로 씁니다.

**순서**
1. 설치
2. 카드 불러오기
3. BM25 (단어 검색)
4. BGE-M3 + FAISS (의미 검색)
5. 리랭커 (후보 다시 줄 세우기)
6. 검색 함수 (두 검색 합치기 + 상품유형 필터 + 리랭커)
7. 평가 (검사기준표 30개 항목, 4가지 방식 비교)
8. 실제 광고 JSON에 적용

**폴더 구조** (팀 레포 `MIX/` 안에 `rag/` 폴더를 두는 기준)
```
MIX/
├─ data/processed/...        ← 주연 님 전처리 결과
└─ rag/
   ├─ 04_rag_search.ipynb    ← 이 노트북
   └─ data/
      ├─ cards.csv           ← 규정카드 탭 CSV
      └─ eval_checks.csv     ← 검사기준표 기반 평가 질문
```

## 1. 설치
처음 한 번만 실행하면 됩니다. 맥에서는 터미널에서 가상환경을 만든 뒤 설치하는 걸 권장합니다 (README 참고).

In [ ]:
# %pip install -q rank_bm25 kiwipiepy sentence-transformers faiss-cpu pandas

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd

RAG_DIR = Path.cwd().resolve()           # 이 노트북이 있는 rag/ 폴더 (노트북을 rag/ 안에서 열어야 함)
REPO_DIR = RAG_DIR.parent                # MIX/ 레포 루트
DATA_DIR = RAG_DIR / 'data'
CACHE_DIR = RAG_DIR / 'cache'; CACHE_DIR.mkdir(exist_ok=True)

## 2. 카드 불러오기
검색에 쓸 텍스트는 **조 제목 + 본문 + 메모**를 합친 것입니다. 메모에는 "가독성 측정 핵심"처럼 우리가 붙인 설명이 있어서 검색에 도움이 됩니다.

In [ ]:
cards = pd.read_csv(DATA_DIR / 'cards.csv', encoding='utf-8-sig').fillna('')
cards['search_text'] = (cards['문서명'] + ' ' + cards['조 제목'] + '\n' + cards['본문'] + '\n' + cards['메모']).str.strip()
print(len(cards), '장')
cards['상품유형'].value_counts()

## 3. BM25 — 단어 검색
한국어는 띄어쓰기 단위로 자르면 "위험고지를", "위험고지가"가 다른 단어가 됩니다. 그래서 형태소 분석기 Kiwi로 명사·동사 어간 같은 의미 있는 조각만 뽑아서 씁니다.

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()
KEEP = ('NNG', 'NNP', 'NR', 'SL', 'SN', 'VV', 'VA', 'XR')   # 일반·고유명사, 수사, 외국어, 숫자, 동사, 형용사, 어근

def tokenize(text):
    return [t.form for t in kiwi.tokenize(text) if t.tag.startswith(KEEP)]

bm25 = BM25Okapi([tokenize(t) for t in cards['search_text']])
print(tokenize('영상 광고에서 위험고지가 짧게만 노출됨'))

## 4. BGE-M3 + FAISS — 의미 검색
카드 190장을 BGE-M3로 숫자 벡터로 바꾸고, 그 벡터를 **FAISS 인덱스**에 넣어 둡니다. FAISS는 "질문 벡터와 가장 가까운 벡터 찾기"를 빠르게 해 주는 벡터 검색 라이브러리입니다.

- 처음 실행할 때 모델(약 2.3GB)을 내려받습니다.
- 인덱스는 `cache/cards.faiss`에 저장돼 다음부터는 바로 불러옵니다. 카드가 바뀌면 자동으로 다시 만듭니다.
- 맥북(Apple Silicon)에서는 `mps`, 코랩 GPU에서는 `cuda`, 그 외에는 `cpu`로 자동 선택됩니다.

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
model = SentenceTransformer('BAAI/bge-m3', device=device)
print('device:', device)

index_path = CACHE_DIR / 'cards.faiss'
ids_path = CACHE_DIR / 'card_ids.json'
if index_path.exists() and ids_path.exists() and json.loads(ids_path.read_text()) == cards['id'].tolist():
    index = faiss.read_index(str(index_path))
else:
    emb = model.encode(cards['search_text'].tolist(), batch_size=8, normalize_embeddings=True, show_progress_bar=True)
    index = faiss.IndexFlatIP(emb.shape[1])      # 정규화한 벡터의 내적 = 코사인 유사도
    index.add(np.asarray(emb, dtype='float32'))
    faiss.write_index(index, str(index_path)); ids_path.write_text(json.dumps(cards['id'].tolist(), ensure_ascii=False))
print('FAISS 인덱스:', index.ntotal, '장')

## 5. 리랭커 — 후보 다시 줄 세우기
BM25와 BGE-M3는 질문과 카드를 **따로** 숫자로 바꿔 비교합니다. 빠르지만 대충 비교하는 방식입니다.

리랭커(`BAAI/bge-reranker-v2-m3`)는 질문과 카드를 **한 쌍으로 같이 읽고** "이 카드가 이 질문의 답인가"를 점수로 매깁니다. 정확하지만 느려서, 앞 단계에서 뽑은 후보 20장에만 씁니다.

```
질문 → BM25 + BGE-M3(FAISS) → 후보 20장 → 리랭커 → 최종 5장
```
처음 실행할 때 모델(약 2.2GB)을 내려받습니다.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=1024)
print('reranker 준비 완료')

## 6. 검색 함수
- **상품유형 필터**: 광고가 투자성이면 `투자성` + `공통` 카드만 후보로 남깁니다. `전체`면 거르지 않습니다.
- **합치기 (RRF)**: 두 검색의 *점수*는 단위가 달라 그대로 더할 수 없어서, *순위*를 씁니다. 각 검색에서 k등이면 `1/(60+k)`점을 주고 더합니다.
- **mode**: `bm25`, `dense`, `hybrid`(둘 합치기), `rerank`(hybrid 후보 20장 → 리랭커)

In [ ]:
def _rank(scores, mask):
    s = np.where(mask, scores, -np.inf)
    order = np.argsort(-s)
    return [i for i in order if mask[i]]

def _dense_scores(query):
    q = model.encode([query], normalize_embeddings=True).astype('float32')
    sims, idx = index.search(q, index.ntotal)        # 전체 카드와의 유사도
    scores = np.full(len(cards), -np.inf); scores[idx[0]] = sims[0]
    return scores

def search(query, product_type='전체', k=5, mode='rerank', rrf_k=60, n_candidates=20):
    if product_type in ('전체', '', None):
        mask = np.ones(len(cards), bool)
    else:
        mask = cards['상품유형'].isin([product_type, '공통']).to_numpy()
    ranks = []
    if mode in ('hybrid', 'rerank', 'bm25'):
        ranks.append(_rank(bm25.get_scores(tokenize(query)), mask))
    if mode in ('hybrid', 'rerank', 'dense'):
        ranks.append(_rank(_dense_scores(query), mask))
    score = {}
    for r in ranks:
        for pos, i in enumerate(r):
            score[i] = score.get(i, 0) + 1 / (rrf_k + pos + 1)
    ordered = sorted(score, key=score.get, reverse=True)
    if mode == 'rerank':
        cand = ordered[:n_candidates]
        rr = reranker.predict([(query, cards['search_text'].iloc[i]) for i in cand], batch_size=8)
        score = {i: float(s) for i, s in zip(cand, rr)}
        ordered = sorted(cand, key=score.get, reverse=True)
    top = ordered[:k]
    out = cards.iloc[top][['id', '조 제목', '상품유형', '본문']].copy()
    out['score'] = [round(score[i], 4) for i in top]
    out['본문'] = out['본문'].str.slice(0, 80) + '…'
    return out.reset_index(drop=True)

search('영상 광고에서 위험고지가 광고 시간 중 짧게만 노출됨', '투자성')

## 7. 평가 — 검사기준표 30개 항목
`eval_checks.csv`는 검사기준표의 항목마다 **광고 상황을 문장으로 쓴 질문**, **핵심 카드(core_id)**, **관련 카드 전체(expected_ids)**를 담고 있습니다.

- **Core@k**: 항목의 핵심 카드가 상위 k개 안에 나온 비율 — 가장 중요한 지표
- **Hit@k**: 관련 카드 중 하나라도 상위 k개 안에 나온 비율
- **Recall@k**: 관련 카드 중 상위 k개 안에 나온 비율 (보조 지표 — 비슷한 조항이 여러 개라 1.0이 안 나와도 괜찮음)

k=5와 k=8을 함께 봅니다. LLM에는 카드 8장 정도는 충분히 넣을 수 있어서, k=8 점수가 높으면 판단 단계에서는 8장을 넘기면 됩니다.

In [ ]:
ev = pd.read_csv(DATA_DIR / 'eval_checks.csv', encoding='utf-8-sig')
ev['expected'] = ev['expected_ids'].str.split(';')

def evaluate(mode, k=5):
    rows = []
    for _, r in ev.iterrows():
        got = search(r['query'], r['product_type'], k=k, mode=mode)['id'].tolist()
        hit = [e for e in r['expected'] if e in got]
        rows.append({'check_id': r['check_id'], 'core': r['core_id'] in got, 'hit': len(hit) > 0,
                     'recall': len(hit) / len(r['expected']),
                     'missed': ';'.join(e for e in r['expected'] if e not in got), 'got': ';'.join(got)})
    return pd.DataFrame(rows)

results = {(m, k): evaluate(m, k) for m in ['bm25', 'dense', 'hybrid', 'rerank'] for k in [5, 8]}
summary = pd.DataFrame({f'{m} @{k}': {'Core': d['core'].mean(), 'Hit': d['hit'].mean(), 'Recall': d['recall'].mean()}
                        for (m, k), d in results.items()}).T.round(3)
summary

핵심 카드를 놓친 항목만 봅니다 (rerank, k=5). 여기 나온 항목은 질문 문장을 고치거나, 카드 메모의 `[검색어]`를 보강하는 대상입니다.

In [ ]:
miss = results[('rerank', 5)]
miss[~miss['core']].merge(ev[['check_id', 'query', 'core_id']], on='check_id')[['check_id', 'query', 'core_id', 'got']]

## 8. 실제 광고 JSON에 적용
주연 님 전처리 JSON(`data/processed/image/json/`, `video/json/`)을 읽어서, **측정값이 검사기준표 기준에 걸리는 항목만** 검색 질문으로 바꿉니다.

- 기준값은 검사기준표의 임시값입니다. 보정하면 `TH`만 고치면 됩니다.
- 상품유형은 JSON에 비어 있는 경우가 많아서 일단 직접 넣습니다 (나중에 브랜드 기준 자동 채우기로 교체).

In [ ]:
TH = {'contrast': 3.0, 'image_pt': 5.9, 'ratio': 0.08, 'exposure_frac': 1/3}

def load_ad(ad_id):
    kind = 'video' if ad_id.startswith('V') else 'image'
    return json.loads((REPO_DIR / f'data/processed/{kind}/json/{ad_id}.json').read_text(encoding='utf-8'))

def build_queries(ad):
    m, qs = ad.get('metrics', {}), []
    is_video = 'duration' in ad
    n_risk = m.get('n_risk_cards') if is_video else sum(p['type'] == 'risk_notice' for p in ad.get('paragraphs', []))
    if not n_risk:
        qs.append(('A1', '위험고지 문구가 광고에서 검출되지 않음. 광고에 반드시 포함해야 하는 위험 고지 사항'))
    c = m.get('risk_contrast_min')
    if c is not None and c < TH['contrast']:
        qs.append(('A2', f'위험고지 글자색이 배경과 비슷해서 잘 안 보임 (명암비 {c:.1f})'))
    if is_video:
        d, e = ad.get('duration') or 0, m.get('risk_exposure_s') or 0
        if d and e / d < TH['exposure_frac']:
            qs.append(('A5', f'영상 광고 {d:.0f}초 중 위험고지가 {e:.1f}초만 노출됨'))
        r = m.get('risk_headline_ratio_min')
    else:
        pt = m.get('mobile_feed', {}).get('risk_median_pt')
        if pt is not None and pt < TH['image_pt']:
            qs.append(('A3', f'인쇄 포스터 광고의 위험고지 글자 크기가 너무 작음 (환산 {pt:.1f}pt)'))
        r = m.get('risk_headline_ratio')
    if r is not None and r < TH['ratio']:
        qs.append(('A4', f'헤드라인 글자에 비해 위험고지 글자가 지나치게 작음 (비율 {r:.2f})'))
    return qs

def run_ad(ad_id, product_type):
    ad = load_ad(ad_id)
    print(f"== {ad_id} {ad.get('brand') or ''} / {product_type}")
    qs = build_queries(ad)
    if not qs:
        print('  측정값 기준으로 걸린 항목 없음 (텍스트 항목 B·C는 05 LLM 단계에서 확인)')
    for cid, q in qs:
        print(f'\n[{cid}] {q}')
        display(search(q, product_type)[['id', '조 제목', 'score']])

run_ad('V101', '투자성')
run_ad('AD01', '투자성')